In [35]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from dotenv import load_dotenv
load_dotenv()

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

from groq import Groq

In [28]:
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=50,
    length_function=len
)

sample_file = 'extracted/iesc110.txt'
loader = TextLoader(sample_file)
documents = loader.load()

chunks = text_splitter.split_documents(documents)

print(f'Total chunks created: {len(chunks)}')
print(f'Average chunk length: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars')
print('\n--- Sample Chunk (first) ---')
print(chunks[0].page_content[:100])

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6289.71it/s]


Total chunks created: 303
Average chunk length: 136 chars

--- Sample Chunk (first) ---
## C hapter 

## **10** 

**==> picture [86 x 85] intentionally omitted <==**

## **WORK AND ENERGY*


In [37]:
db_path = 'data/vector_db/qwen-3-embedding-faiss_index'

if os.path.exists(db_path):
    print('Loading existing FAISS database from disk...')
    vector_store = FAISS.load_local(db_path, embeddings, allow_dangerous_deserialization=True)
else:
    print('Building FAISS database from extracted files...')
    documents = []
    
    extracted_dir = 'extracted'
    for f in os.listdir(extracted_dir):
        if f.endswith('.txt'):
            file_path = os.path.join(extracted_dir, f)
            
            loader = TextLoader(file_path)
            docs = loader.load()
            
            # Update metadata for each document
            for doc in docs:
                doc.metadata.update({
                    "chapter": f.replace('_paragraphs.txt', ''),
                    "filename": f
                })
            
            documents.extend(docs)
    
    print(f'Total documents loaded: {len(documents)}')
    chunks = text_splitter.split_documents(documents)
    print(f'Total chunks created: {len(chunks)}')
    
    # Debug: Check if chunks have content
    if chunks:
        print(f'Sample chunk content length: {len(chunks[0].page_content)}')
        print(f'Sample chunk preview: {chunks[0].page_content[:100]}...')
    else:
        print('No chunks created!')
    
    vector_store = FAISS.from_documents(chunks, embeddings)
    
    vector_store.save_local(db_path)
    print(f'Database saved to {db_path}')

print(f'Total documents in vector store: {len(vector_store.docstore._dict)}')

Building FAISS database from extracted files...
Total documents loaded: 14
Total chunks created: 3428
Sample chunk content length: 101
Sample chunk preview: ## C hapter **4** 

**==> picture [86 x 86] intentionally omitted <==**

## **STRUCTURE OF THE ATOM*...
Database saved to data/vector_db/qwen-3-embedding-faiss_index
Total documents in vector store: 3428


In [ ]:
test_queries = [
    'What are the three states of matter?',
    'What is the difference between pure substances and mixtures?',
    'State the law of conservation of mass.',
    'What are the sub-atomic particles in an atom?'
]

def retrieve_with_langchain(query, k=3):
    """Retrieve documents using LangChain similarity search"""
    results = vector_store.similarity_search_with_score(query, k=k)
    retrieved_docs = []
    
    for doc, score in results:
        retrieved_docs.append({
            'text': doc.page_content,
            'chapter': doc.metadata.get('chapter', 'unknown'),
            'filename': doc.metadata.get('filename', 'unknown'),
            'score': score
        })
    
    return retrieved_docs

for query in test_queries:
    print(f'\nQuery: {query}')
    results = retrieve_with_langchain(query, k=3)
    for i, r in enumerate(results, 1):
        print(f'  [{i}] Chapter: {r["chapter"]} | File: {r["filename"]}')
        print(f'      Score: {r["score"]:.4f}')
        print(f'      {r["text"][:120]}...')
    print('-' * 60)


Query: What are the three states of matter?
  [1] Chapter: iesc101 | File: iesc101_paragraphs.txt
      Score: 0.4119
      can be seen and compared in the three states of matter._...
  [2] Chapter: iesc101 | File: iesc101_paragraphs.txt
      Score: 0.4371
      ## **1.4 Can Matter Change its State?**

We all know from our observation that water can exist in three states of matter...
  [3] Chapter: iesc101 | File: iesc101_paragraphs.txt
      Score: 0.4724
      Now, let us study about the properties of these three states of matter in detail.

## **1.3.1 THE SOLID STATE**

## _**A...
------------------------------------------------------------

Query: What is the difference between pure substances and mixtures?
  [1] Chapter: iesc101 | File: iesc101_paragraphs.txt
      Score: 0.6824
      about the nature of matter — is it continuous or particulate?...
  [2] Chapter: iesc101 | File: iesc101_paragraphs.txt
      Score: 0.8256
      _2. (a) Tabulate the differences in the characterisi

In [ ]:
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
GROUNDING_PROMPT = """
You are a study assistant for PariShiksha. 
Use ONLY the context provided below to answer the question.
If the answer is not present in the context, respond with:
"This question is outside the provided NCERT content."
Do not infer, extrapolate, or use outside knowledge.

Context:
{context}

Question: {question}
Answer:
"""

def answer(question, k=3):
    """Answer question using LangChain retrieval and Groq Llama 3.1 Instant"""
    # Retrieve relevant documents using LangChain
    retrieved_docs = retrieve_with_langchain(question, k=k)
    context = '\n\n---\n\n'.join([doc['text'] for doc in retrieved_docs])
    prompt = GROUNDING_PROMPT.format(context=context, question=question)
    
    # Use Groq Llama 3.1 Instant for inference
    client = Groq(api_key=GROQ_API_KEY)
    completion = client.chat.completions.create(
        model='llama-3.1-8b-instant',  # Llama 3.1 Instant from Groq
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0
    )
    
    return {
        'answer': completion.choices[0].message.content,
        'retrieved_docs': retrieved_docs
    }

result = answer('What are the three states of matter?')
print(f'Answer: {result["answer"]}')
print(f'\nSources: {[c["chapter"] for c in result["retrieved_docs"]]}')

Answer: The three states of matter are:

1. solid, as ice
2. liquid, as the familiar water
3. (missing information, but the context implies the third state is mentioned in the next section)

Sources: ['iesc101', 'iesc101', 'iesc101']


In [34]:
oos_result = answer('Explain quantum entanglement from Chapter 9')
print(f'Out-of-scope test:')
print(f'Answer: {oos_result["answer"]}')
print(f'Sources: {[c["chapter"] for c in oos_result["retrieved_docs"]]}')

Out-of-scope test:
Answer: "This question is outside the provided NCERT content."
Sources: ['iesc101', 'iesc101', 'iesc101']


The PariShiksha RAG pipeline has been successfully updated to use:
- **LangChain** for document processing and vector store management
- **OpenAI embeddings** for semantic similarity search
- **FAISS** for efficient vector storage and retrieval
- **Groq's Llama 3.1 Instant** for fast inference

### Evaluation

In [ ]:
import json
with open('data/eval_questions.json', 'r') as f:
    categories = json.load(f)

total_questions = sum(len(cat['questions']) for cat in categories)
print(f'Evaluation set: {total_questions} questions')
for cat in categories:
    print(f'  {cat["category"]} ({cat["type"]}): {len(cat["questions"])} questions')

# Run evaluation
results = []

for cat in categories:
    q_type = cat['type']
    for question in cat['questions']:
        res = answer(question)
        ans = res['answer']
        is_refusal = 'outside' in ans.lower() or 'not present' in ans.lower() or 'not in the context' in ans.lower()
        
        if q_type == 'out_of_scope':
            correctness = 'yes' if is_refusal else 'no'
            grounded = 'yes' if is_refusal else 'no'
            refusal = 'yes' if is_refusal else 'no'
        else:
            correctness = 'yes' if not is_refusal and len(ans) > 20 else ('no' if is_refusal else 'partial')
            grounded = 'yes' if not is_refusal else 'no'
            refusal = 'na' if not is_refusal else 'no'
        
        results.append({
            'question': question, 'type': q_type, 'answer': ans,
            'correctness': correctness, 'grounded': grounded, 'refusal': refusal
        })

# Summary
correct = sum(1 for r in results if r['correctness'] == 'yes')
grounded = sum(1 for r in results if r['grounded'] == 'yes')
print(f'\nResults: {correct}/{len(results)} correct, {grounded}/{len(results)} grounded')

# Display results table
print(f'{"#":<3} {"Type":<15} {"Correct":<10} {"Grounded":<10} {"Question":<55}')
print('-' * 95)
for i, r in enumerate(results, 1):
    print(f'{i:<3} {r["type"]:<15} {r["correctness"]:<10} {r["grounded"]:<10} {r["question"][:55]}')

Evaluation set: 20 questions
  Direct Textbook (direct): 12 questions
  Paraphrased (paraphrased): 3 questions
  Out of Scope (out_of_scope): 5 questions

Results: 18/20 correct, 18/20 grounded
#   Type            Correct    Grounded   Question                                               
-----------------------------------------------------------------------------------------------
1   direct          yes        yes        What are the three states of matter?
2   direct          yes        yes        Why is ice at 273 K more effective in cooling than wate
3   direct          yes        yes        What produces more severe burns, boiling water or steam
4   direct          yes        yes        Calculate the molecular mass of water (H2O).
5   direct          yes        yes        What is the powerhouse of the cell and why?
6   direct          yes        yes        What is the difference between a plant cell and an anim
7   direct          no         no         Define displacement and 

In [43]:
print("=== VECTOR STORE CONTENT ANALYSIS ===")

# Sample some chunks to understand content quality
sample_docs = list(vector_store.docstore._dict.values())[:2]
for i, doc in enumerate(sample_docs, 1):
    print(f"\nSample {i}:")
    print(f"  Metadata: {doc.metadata}")
    print(f"  Content: {doc.page_content[:200]}...")
    print(f"  Length: {len(doc.page_content)} chars")

print(f"\nTotal documents: {len(vector_store.docstore._dict)}")
unique_files = set(doc.metadata.get('filename', 'unknown') for doc in vector_store.docstore._dict.values())
print(f"Unique files: {unique_files}")

test_specific = retrieve_with_langchain("protons neutrons electrons", k=3)
print(f"\n=== SPECIFIC QUERY TEST ===")
for i, r in enumerate(test_specific, 1):
    print(f"  [{i}] Score: {r['score']:.4f}")
    print(f"      {r['text'][:150]}...")

=== VECTOR STORE CONTENT ANALYSIS ===

Sample 1:
  Metadata: {'source': 'extracted/iesc104.txt', 'chapter': 'iesc104.txt', 'filename': 'iesc104.txt'}
  Content: ## C hapter **4** 

**==> picture [86 x 86] intentionally omitted <==**

## **STRUCTURE OF THE ATOM**...
  Length: 101 chars

Sample 2:
  Metadata: {'source': 'extracted/iesc104.txt', 'chapter': 'iesc104.txt', 'filename': 'iesc104.txt'}
  Content: In Chapter 3, we have learnt that atoms and molecules are the fundamental building blocks of matter. The existence of different kinds of matter is due to different atoms...
  Length: 169 chars

Total documents: 3428
Unique files: {'iesc103.txt', 'iesc107.txt', 'iesc109.txt', 'iesc108.txt', 'iesc102.txt', 'iesc110.txt', 'iesc101.txt', 'iesc112.txt', 'iesc105.txt', 'iesc106.txt', 'iesc1ps.txt', 'iesc104.txt', 'iesc111.txt', 'iesc1an.txt'}

=== SPECIFIC QUERY TEST ===
  [1] Score: 0.3851
      (i) electrons, (ii) protons and (iii) neutrons. Electrons are negatively charged, protons are p